In [1]:
!pip install kagglehub
!pip install transformers
!pip install scikit-learn
!pip install datasets
!pip install ftfy
!pip install datasets
import requests
import yaml
import re
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification,AutoModelForMaskedLM, Trainer, TrainingArguments,  DataCollatorForLanguageModeling
import torch
import numpy as np
import pandas as pd
from ftfy import fix_text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00


In [2]:
import pandas as pd

df = pd.read_csv("news_train_only.csv")
df['title'] = df['title'].apply(lambda x: fix_text(str(x)))
full_list = df[df["important"] < 1]["title"].dropna().unique().tolist()

train_text = full_list[:len(full_list)//2]



In [3]:
tokenizer = AutoTokenizer.from_pretrained("./")
model = AutoModelForMaskedLM.from_pretrained("./")



Some weights of BertForMaskedLM were not initialized from the model checkpoint at ./ and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
from transformers import BertForMaskedLM, BertForSequenceClassification

cls_model = BertForSequenceClassification.from_pretrained(".")
mlm_model = BertForMaskedLM.from_pretrained("yiyanghkust/finbert-tone")
cls_state = cls_model.bert.state_dict()

filtered_state = {k: v for k, v in cls_state.items() if not k.startswith("pooler.")}

missing, unexpected = mlm_model.bert.load_state_dict(filtered_state, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of BertForMaskedLM were not initialized from the model checkpoint at yiyanghkust/finbert-tone and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Missing keys: []
Unexpected keys: []


In [5]:
mlm_dataset = Dataset.from_dict({"text": train_text})

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = mlm_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/95396 [00:00<?, ? examples/s]

In [6]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

training_args = TrainingArguments(
    output_dir="./",
    per_device_train_batch_size=16,
    num_train_epochs=2,
    learning_rate=5e-6,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss
500,7.369400
1000,6.191900
1500,5.744300
2000,5.500000
2500,5.341600
3000,5.179600
3500,5.093100
4000,4.985200
4500,4.919600
5000,4.874000


TrainOutput(global_step=11926, training_loss=4.975190125074679, metrics={'train_runtime': 6707.9318, 'train_samples_per_second': 28.443, 'train_steps_per_second': 1.778, 'total_flos': 1.2554394592352256e+16, 'train_loss': 4.975190125074679, 'epoch': 2.0})

In [7]:
model.save_pretrained("./finbert_domain_adapted")
tokenizer.save_pretrained("./finbert_domain_adapted")


('./finbert_domain_adapted/tokenizer_config.json',
 './finbert_domain_adapted/special_tokens_map.json',
 './finbert_domain_adapted/vocab.txt',
 './finbert_domain_adapted/added_tokens.json',
 './finbert_domain_adapted/tokenizer.json')